In [15]:
import warnings
import os
warnings.filterwarnings('ignore')

In [16]:
import os
import logging
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()

In [17]:
!pip install -q langchain langchain-core langchain-community langchain-groq --quiet

In [18]:
!pip install requests==2.32.4 --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [19]:
import os
from google.colab import userdata

# Load API key
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("API Key Loaded:", "YES" if os.environ.get("GROQ_API_KEY") else "NO")

API Key Loaded: YES


###1. Basic LLM Call

In [20]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # fast + free
    temperature=0
)

response = llm.invoke("Explain LangChain in one line")
print(response.content)

LangChain is an open-source framework that enables developers to build applications powered by large language models, providing a set of tools and APIs to interact with and integrate AI models into various projects.


###2. PromptTemplate usage

In [21]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Explain {topic} in simple terms"
)

formatted_prompt = prompt.format(topic="LangChain")

print(formatted_prompt)
print("\n--- LLM Output ---\n")

# Using the 'llm' instance defined in previous cells
print(llm.invoke(formatted_prompt).content)

Explain LangChain in simple terms

--- LLM Output ---

LangChain is a framework that helps developers build applications using large language models (LLMs) like chatbots, virtual assistants, or other AI-powered tools. 

Imagine you have a super smart, magic notebook that can understand and respond to anything you write in it. This notebook is like a large language model. However, to make the notebook really useful, you need to teach it how to do specific tasks, like answering questions, generating text, or even controlling other devices.

LangChain is like a set of instructions that helps you teach the magic notebook (or the large language model) how to do these tasks. It provides a way to break down complex tasks into smaller, manageable pieces, and then use the language model to complete each piece.

The "chain" part of LangChain refers to the idea of linking together multiple language models, or using a single model in a series of steps, to achieve a specific goal. This allows devel

###3. Simple Chain

In [22]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"topic": "AI Agents"})
print(result)

**What is an AI Agent.** 
An AI agent is a computer program that can perform tasks on its own, using artificial intelligence (AI) to make decisions and take actions. Think of it like a robot that can think and act for itself.

**Key Characteristics:**

1. **Autonomy**: AI agents can work independently, without human intervention.
2. **Decision-making**: They can make decisions based on the information they have and the goals they're trying to achieve.
3. **Action**: They can take actions to achieve their goals, like moving, communicating, or solving problems.
4. **Learning**: Many AI agents can learn from their experiences and improve over time.

**Examples of AI Agents:**

1. **Virtual Assistants**: Like Siri, Alexa, or Google Assistant, which can understand voice commands and perform tasks.
2. **Self-Driving Cars**: Which use AI to navigate roads, avoid obstacles, and make decisions.
3. **Chatbots**: That can have conversations with humans, answer questions, and provide customer supp

###4. Agent with tool

In [23]:
!pip install -q langchain langgraph langchain-groq

import os
from google.colab import userdata

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# Set your Groq API key from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Create Groq model
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Define tools

@tool
def convert_usd_to_inr(usd: float) -> str:
    """Convert USD to INR using a fixed example exchange rate of 1 USD = 83 INR."""
    inr = usd * 83
    return f"${usd:.2f} is approximately ₹{inr:.2f}"

@tool
def calculate_emi(principal: float, annual_rate: float, months: int) -> str:
    """Calculate monthly EMI for a loan using principal, annual interest rate, and loan duration in months."""
    monthly_rate = annual_rate / 12 / 100
    if monthly_rate == 0:
        emi = principal / months
    else:
        emi = principal * monthly_rate * (1 + monthly_rate) ** months / ((1 + monthly_rate) ** months - 1)
    total_payment = emi * months
    return (
        f"Monthly EMI: ₹{emi:.2f}, "
        f"Total Payment: ₹{total_payment:.2f}"
    )

tools = [convert_usd_to_inr, calculate_emi]

# Create ReAct agent
agent = create_react_agent(
    model=model,
    tools=tools
)

# Run the agent
result = agent.invoke({
    "messages": [
        (
            "human",
            "Convert 250 USD to INR and calculate the EMI for a ₹500000 loan at 10% annual interest for 24 months."
        )
    ]
})

print("\n=== Final Answer ===")
print(result["messages"][-1].content)


=== Final Answer ===
The conversion of 250 USD to INR is approximately ₹20750.00. The monthly EMI for a ₹500000 loan at 10% annual interest for 24 months is approximately ₹23072.46, with a total payment of ₹553739.12.


###5. Memory example

In [24]:
!pip install -qU langchain langchain-openai langchain-community --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [25]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

# 1. Define a prompt that includes a placeholder for chat history
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# 2. Build the chain using LCEL (the '|' pipe operator)
chain = prompt | model | StrOutputParser()

# 3. Use a dictionary to store history (In-memory)
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 4. Wrap the chain with history management
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "user_1"}}

# Interaction 1
resp1 = with_message_history.invoke({"input": "Hi, my name is Sai Kadiravan."}, config=config)
print(f"Assistant: {resp1}")

# Interaction 2 (Testing memory)
resp2 = with_message_history.invoke({"input": "What is my name?"}, config=config)
print(f"Assistant: {resp2}")

Assistant: Hello Sai Kadiravan, it's nice to meet you. Is there something I can help you with or would you like to chat?
Assistant: Your name is Sai Kadiravan.


###Vector Store Implementation (FAISS + HuggingFace)

In [26]:
!pip install -qU langchain-groq langchain-huggingface langchain-community docarray wikipedia sentence-transformers --quiet

In [27]:
# 1. Install necessary libraries for Vector Stores
!pip install -qU langchain-huggingface faiss-cpu

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Define the raw data (similar to your structure)
raw_docs = [
    Document(page_content="LangChain provides tools for building agents and managing memory.",
             metadata={"source": "intro.txt"}),
    Document(page_content="FAISS is a library for fast similarity search over dense vectors.",
             metadata={"source": "vector_info.txt"}),
    Document(page_content="The Model Context Protocol (MCP) connects AI models to local data.",
             metadata={"source": "mcp_info.txt"}),
]

# 3. Text Splitting
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_documents(raw_docs)
print(f"Created {len(chunks)} chunks from {len(raw_docs)} documents.")

# 4. Initialize Local Embeddings (Free, runs on Colab CPU)
# Using 'all-MiniLM-L6-v2' - lightweight and highly accurate for its size
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 5. Build the Vector Store (FAISS)
vector_store = FAISS.from_documents(chunks, embeddings)
print(" FAISS Vector store built successfully.")
print(f"Total vectors indexed: {vector_store.index.ntotal}")

# 6. Manual Similarity Search (To test the Vector Store independently)
query = "What is FAISS?"
docs = vector_store.similarity_search(query, k=1)

print("\n--- Similarity Search Result ---")
for doc in docs:
    print(f"Source: {doc.metadata['source']}")
    print(f"Content: {doc.page_content}")

Created 3 chunks from 3 documents.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 FAISS Vector store built successfully.
Total vectors indexed: 3

--- Similarity Search Result ---
Source: vector_info.txt
Content: FAISS is a library for fast similarity search over dense vectors.


###Real-World Use Case: Personal Technical Library Assistant

In [28]:
# Real-World Use Case: RAG-based Technical Assistant
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

# 1. Setup the Model (Updated to supported model) & Embeddings
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. The Knowledge Base (The "Real World" Data)
tech_docs = [
    Document(page_content="The Model Context Protocol (MCP) allows AI agents to interact with local tools safely."),
    Document(page_content="In NLP, Token Classification is used for Named Entity Recognition (NER)."),
    Document(page_content="LangChain's LCEL allows for declarative composition of chains.")
]

# 3. Component: Indexing (Splitting + Vector Store)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
doc_chunks = text_splitter.split_documents(tech_docs)
vector_db = FAISS.from_documents(doc_chunks, embeddings)

# 4. Component: The Chain (Orchestration)
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# Modern LCEL Chain
rag_chain = (
    {"context": vector_db.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Execution
print("--- Real World Assistant ---")
try:
    response = rag_chain.invoke("How does MCP help AI agents?")
    print(f"Response: {response}")
except Exception as e:
    print(f"Error: {e}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--- Real World Assistant ---
Response: The Model Context Protocol (MCP) allows AI agents to interact with local tools safely.
